This script generates synthetic records across all six tables, preserving realistic physical mass balance, downtime distributions, polymer mechanics, and compliance formulas.

In [1]:
import random
from datetime import datetime, timedelta
import pandas as pd

In [2]:
random.seed(42)

In [3]:
# ============================================================================
# 1. INBOUND WEIGHBRIDGE
# ============================================================================
suppliers = ['SUP-PANDA-WASTE', 'SUP-BEAUPARC-IRE', 'SUP-BIFFA-LOGISTICS', 'SUP-TESCO-DC', 'SUP-AGRI-RECYCLE']
haulers = ['FREIGHT-EXPRESS', 'TRANS-IRISH-LOG', 'GREEN-LINE-LOG', 'DIRECT-HAUL']
polymers = ['LDPE', 'HDPE']

weighbridge_data = []
base_date = datetime(2026, 9, 1, 6, 30)

for ticket_id in range(101, 161):  # 60 delivery trucks
    supplier = random.choice(suppliers)
    polymer = random.choice(polymers)
    tare = round(random.uniform(13800, 14500), 2)
    net = round(random.uniform(11000, 17000), 2)
    gross = tare + net
    
    # Specific suppliers have dirtier baseline loads
    if supplier in ['SUP-BEAUPARC-IRE', 'SUP-AGRI-RECYCLE']:
        contamination = round(random.uniform(6.5, 14.8), 2)
    else:
        contamination = round(random.uniform(1.2, 4.9), 2)
        
    price_per_tonne = 440.00 if polymer == 'HDPE' else 415.00
    arrival = base_date + timedelta(hours=(ticket_id - 101) * 1.5, minutes=random.randint(0, 40))
    
    weighbridge_data.append({
        'ticket_id': ticket_id,
        'supplier_id': supplier,
        'hauler_id': random.choice(haulers),
        'arrival_time': arrival,
        'polymer_type': polymer,
        'gross_weight_kg': gross,
        'tare_weight_kg': tare,
        'sample_contamination_pct': contamination,
        'price_per_tonne': price_per_tonne
    })

df_weighbridge = pd.DataFrame(weighbridge_data)

In [4]:
# ============================================================================
# 2. WASH PLANT BATCHES
# ============================================================================
wash_data = []
batch_id_counter = 501

for _, ticket in df_weighbridge.iterrows():
    net_weight = ticket['gross_weight_kg'] - ticket['tare_weight_kg']
    # Split large delivery into 2 batches, smaller delivery into 1
    num_batches = 2 if net_weight > 13500 else 1
    split_mass = round(net_weight / num_batches, 2)
    
    for i in range(num_batches):
        line = 'LINE_1' if ticket['polymer_type'] == 'LDPE' else 'LINE_2'
        start_time = ticket['arrival_time'] + timedelta(hours=1 + (i * 3.5), minutes=random.randint(5, 25))
        end_time = start_time + timedelta(hours=3, minutes=random.randint(15, 45))
        
        feed_mass = split_mass
        # Contamination directly increases reject waste and decreases clean output
        shrinkage_rate = (ticket['sample_contamination_pct'] / 100.0) + random.uniform(0.06, 0.10)
        clean_output = round(feed_mass * (1.0 - shrinkage_rate), 2)
        rejects = round((feed_mass - clean_output) * random.uniform(0.75, 0.88), 2)
        
        # Water & Energy intensity
        water_m3 = round((feed_mass / 1000.0) * random.uniform(1.8, 2.6), 2)
        kwh_factor = 0.32 if line == 'LINE_1' else 0.42
        energy_kwh = round(clean_output * random.uniform(kwh_factor * 0.95, kwh_factor * 1.25), 2)
        
        wash_data.append({
            'batch_id': batch_id_counter,
            'ticket_id': ticket['ticket_id'],
            'line_id': line,
            'start_time': start_time,
            'end_time': end_time,
            'feed_mass_kg': feed_mass,
            'clean_flake_output_kg': clean_output,
            'reject_waste_kg': rejects,
            'water_consumption_m3': water_m3,
            'energy_kwh': energy_kwh
        })
        batch_id_counter += 1

df_wash = pd.DataFrame(wash_data)

In [5]:
# ============================================================================
# 3. COMPOUNDING & EXTRUSION
# ============================================================================
extrusion_data = []
extrusion_id_counter = 701

for _, wash in df_wash.iterrows():
    flake_input = wash['clean_flake_output_kg']
    # Formulate recipe: virgin plastic blend ratio (5% to 20%)
    virgin_ratio = random.choice([0.05, 0.10, 0.15, 0.20])
    virgin_kg = round(flake_input * (virgin_ratio / (1.0 - virgin_ratio)), 2)
    masterbatch_kg = round((flake_input + virgin_kg) * 0.02, 2)  # 2% color/stabilizer additive
    
    total_charge = flake_input + virgin_kg + masterbatch_kg
    # Extrusion purge and screen loss (~1.5% to 3.0%)
    melt_loss_pct = random.uniform(0.015, 0.030)
    pellet_yield = round(total_charge * (1.0 - melt_loss_pct), 2)
    
    start_time = wash['end_time'] + timedelta(hours=1, minutes=random.randint(10, 30))
    end_time = start_time + timedelta(hours=4, minutes=random.randint(0, 30))
    
    # Dirty input causes pressure spikes across continuous screen filters
    base_pressure = 135.0 if wash['reject_waste_kg'] < 600 else 185.0
    screen_pressure = round(base_pressure + random.uniform(5.0, 45.0), 1)
    melt_temp = round(random.uniform(195.0, 218.0), 1)
    extruder_kwh = round(pellet_yield * random.uniform(0.40, 0.52), 2)
    
    extrusion_data.append({
        'extrusion_id': extrusion_id_counter,
        'batch_id': wash['batch_id'],
        'extruder_id': random.choice(['EXT-01', 'EXT-02']),
        'start_time': start_time,
        'end_time': end_time,
        'flake_input_kg': flake_input,
        'virgin_blend_ratio_pct': round(virgin_ratio * 100, 2),
        'virgin_input_kg': virgin_kg,
        'masterbatch_kg': masterbatch_kg,
        'melt_temp_c': melt_temp,
        'screen_filter_pressure_bar': screen_pressure,
        'pellet_yield_kg': pellet_yield,
        'extruder_kwh': extruder_kwh
    })
    extrusion_id_counter += 1

df_extrusion = pd.DataFrame(extrusion_data)

In [6]:
# ============================================================================
# 4. QA LAB TESTS
# ============================================================================
qa_data = []
test_id_counter = 1001

for _, ext in df_extrusion.iterrows():
    test_time = ext['end_time'] + timedelta(minutes=random.randint(15, 45))
    
    # Simulate realistic lab metrics
    mfi = round(random.uniform(0.65, 1.45), 2)
    density = round(random.uniform(0.9210, 0.9280), 4)
    moisture = round(random.uniform(0.018, 0.075), 3)  # > 0.05% causes brittle bubble
    tensile = round(random.uniform(16.5, 26.0), 2)
    
    if moisture > 0.055 or mfi > 1.35 or tensile < 17.5:
        status = 'REJECTED' if moisture > 0.07 else 'DOWNGRADED'
    else:
        status = 'PASSED'
        
    qa_data.append({
        'test_id': test_id_counter,
        'extrusion_id': ext['extrusion_id'],
        'test_timestamp': test_time,
        'melt_flow_index': mfi,
        'density_g_cm3': density,
        'moisture_pct': moisture,
        'tensile_break_strength_mpa': tensile,
        'qa_status': status
    })
    test_id_counter += 1

df_qa = pd.DataFrame(qa_data)

In [7]:
# ============================================================================
# 5. BAG PRODUCTION (CONVERSION)
# ============================================================================
production_data = []
work_order_counter = 9001
sku_catalog = {
    'BAG-REFUSE-85L': {'unit_mass_kg': 0.18, 'scrap_base': 0.035},
    'BAG-WHEELIE-240L': {'unit_mass_kg': 0.38, 'scrap_base': 0.040},
    'BAG-RUBBLE-HEAVY': {'unit_mass_kg': 0.65, 'scrap_base': 0.055}
}
downtime_causes = ['SEAL_FAILURE', 'BUBBLE_BURST', 'CHANGEOVER', 'KNIFE_BLUNT', 'HEATER_FAULT']
downtime_weights = [0.42, 0.32, 0.14, 0.08, 0.04]

for _, ext in df_extrusion.iterrows():
    # Retrieve matching QA status
    matching_qa = df_qa[df_qa['extrusion_id'] == ext['extrusion_id']].iloc[0]
    
    # Rejected batches are not converted into finished goods
    if matching_qa['qa_status'] == 'REJECTED':
        continue
        
    sku_key = random.choice(list(sku_catalog.keys()))
    sku_spec = sku_catalog[sku_key]
    
    start_time = ext['end_time'] + timedelta(hours=random.randint(2, 6))
    end_time = start_time + timedelta(hours=random.randint(5, 8))
    
    pellet_input = ext['pellet_yield_kg']
    # Higher moisture or downgraded resin creates elevated scrap
    scrap_rate = sku_spec['scrap_base'] + (0.04 if matching_qa['qa_status'] == 'DOWNGRADED' else 0.01)
    scrap_kg = round(pellet_input * scrap_rate, 2)
    roll_mass = round(pellet_input - scrap_kg, 2)
    units_produced = int(roll_mass / sku_spec['unit_mass_kg'])
    
    downtime_min = random.randint(15, 60) if matching_qa['qa_status'] == 'PASSED' else random.randint(70, 165)
    cause = random.choices(downtime_causes, weights=downtime_weights)[0]
    
    production_data.append({
        'work_order_id': work_order_counter,
        'extrusion_id': ext['extrusion_id'],
        'machine_id': random.choice(['BAG_M01', 'BAG_M02', 'BAG_M03']),
        'sku_id': sku_key,
        'start_time': start_time,
        'end_time': end_time,
        'pellet_input_kg': pellet_input,
        'units_produced': units_produced,
        'total_roll_mass_kg': roll_mass,
        'scrap_kg': scrap_kg,
        'downtime_minutes': downtime_min,
        'downtime_reason': cause
    })
    work_order_counter += 1

df_production = pd.DataFrame(production_data)

In [8]:
# ============================================================================
# 6. OUTBOUND DISPATCH & ESG COMPLIANCE
# ============================================================================
customers = ['CUST-DUBLIN-COUNCIL', 'CUST-TESCO-IE', 'CUST-BUNZL-CLEAN', 'CUST-MONAGHAN-COUNTY', 'CUST-BWG-FOODS']
dispatch_data = []
dispatch_id_counter = 4001

for _, prod in df_production.iterrows():
    ext_parent = df_extrusion[df_extrusion['extrusion_id'] == prod['extrusion_id']].iloc[0]
    
    # Calculate exact Certified Post-Consumer Recycled (PCR) content percentage
    total_recipe_mass = ext_parent['flake_input_kg'] + ext_parent['virgin_input_kg'] + ext_parent['masterbatch_kg']
    certified_pcr = round((ext_parent['flake_input_kg'] / total_recipe_mass) * 100.0, 2)
    
    ship_date = prod['end_time'] + timedelta(days=random.randint(1, 4), hours=random.randint(1, 5))
    units_shipped = prod['units_produced']
    mass_shipped = prod['total_roll_mass_kg']
    
    # Avoided CO2e calculation: 1 kg PCR PE displaces ~1.85 kg CO2e vs. virgin fossil polymer
    avoided_co2e = round(mass_shipped * (certified_pcr / 100.0) * 1.85, 2)
    unit_price = 0.35 if '85L' in prod['sku_id'] else (0.75 if '240L' in prod['sku_id'] else 1.20)
    invoice_val = round(units_shipped * unit_price, 2)
    
    dispatch_data.append({
        'dispatch_id': dispatch_id_counter,
        'work_order_id': prod['work_order_id'],
        'customer_id': random.choice(customers),
        'delivery_note_ref': f"DN-2026-{dispatch_id_counter}",
        'ship_date': ship_date,
        'units_shipped': units_shipped,
        'net_mass_shipped_kg': mass_shipped,
        'certified_pcr_pct': certified_pcr,
        'avoided_co2e_kg': avoided_co2e,
        'invoice_value_eur': invoice_val
    })
    dispatch_id_counter += 1

df_dispatch = pd.DataFrame(dispatch_data)

In [9]:
# Export all 6 tables to CSV
df_weighbridge.to_csv('weighbridge_intake.csv', index=False)
df_wash.to_csv('wash_batches.csv', index=False)
df_extrusion.to_csv('extrusion_compounding.csv', index=False)
df_qa.to_csv('qa_lab_tests.csv', index=False)
df_production.to_csv('bag_production.csv', index=False)
df_dispatch.to_csv('outbound_dispatch.csv', index=False)

print("Full Enterprise Dataset Generated Successfully:")
print(f"1. weighbridge_intake     : {len(df_weighbridge)} rows")
print(f"2. wash_batches           : {len(df_wash)} rows")
print(f"3. extrusion_compounding  : {len(df_extrusion)} rows")
print(f"4. qa_lab_tests           : {len(df_qa)} rows")
print(f"5. bag_production         : {len(df_production)} rows")
print(f"6. outbound_dispatch      : {len(df_dispatch)} rows")

Full Enterprise Dataset Generated Successfully:
1. weighbridge_intake     : 60 rows
2. wash_batches           : 90 rows
3. extrusion_compounding  : 90 rows
4. qa_lab_tests           : 90 rows
5. bag_production         : 85 rows
6. outbound_dispatch      : 85 rows
